# Grain Strategy Research Thinking Process

This notebook is not a final-strategy runner. It is organized as separate runnable code blocks that show the research logic step by step.

The goal is to make the thinking process visible:

1. Start with simple baselines.
2. Test whether family weighting helps.
3. Test whether IC selection helps or overfits.
4. Test whether regimes help.
5. Test fitted models as benchmarks, not preferred solutions.
6. Use performance analysis to diagnose weak periods.
7. Only then test product-specific improvements.

Products covered:
- Soybeans
- Corn
- Wheat SRW / HRW

In [3]:
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 240)

DATA_DIR = "train_set"
COST_ONLY = True

## 0. Load Shared Experiment Functions

This cell imports the actual experiment modules used during the research. The notebook does not invent new logic; it exposes the existing research path in clean blocks.

In [5]:
from family_regime_model_comparison import run_family_regime_model_comparison
from linear_online_model_experiment import run_linear_online_model_experiment
from soybean_external_signal_experiment import (
    run_soybean_given_only_experiment,
    run_soybean_external_signal_experiment,
)
from soybean_low_vol_switch_experiment import run_soybean_low_vol_switch_experiment
from corn_external_signal_experiment import run_corn_signal_experiment
from corn_abundant_supply_improvement import run_corn_abundant_supply_improvement
from wheat_improvement_experiment import run_wheat_improvement_experiment

from grain_futures_strategy import period_performance

# Part 1: Generic Provided-Data Audit

Before building product-specific strategies, I tested whether a generic rule worked across products using only the provided data.

This answers:

- Is simple averaging enough?
- Does equal family weighting help?
- Does IC family selection help?
- Do trend/MR regimes help?
- Do OLS/Kalman/Ridge fitted models help?

## 1. Average All Signals and Equal Family Weight

Rationale:

Start with the lowest-overfit possible tests. If these work, I do not need more complex selection.

Tests:

- `avg_all_signals`: simple average of every provided economic signal.
- `equal_family_weight`: average signals within each family, then equal-weight the families.

In [6]:
family_out = run_family_regime_model_comparison(data_dir=DATA_DIR)
family_results = family_out["results"].copy()

if COST_ONLY:
    family_results = family_results.loc[family_results["cost_adjusted"].astype(bool)].copy()

baseline_rows = family_results.loc[
    family_results["strategy"].isin(["avg_all_signals", "equal_family_weight"]),
    [
        "commodity", "strategy", "mode", "cost_adjusted", "oos_sharpe", "oos_pnl", "oos_dd",
        "full_sharpe", "full_dd", "hit_rate", "win_days", "turnover", "overfit_comment",
    ],
].sort_values(["commodity", "strategy"])

baseline_rows

,commodity,strategy,mode,cost_adjusted,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,hit_rate,win_days,turnover,overfit_comment
9,CORN,avg_all_signals,long_short,True,-0.222904,-92.230612,-436.504097,0.050660,-439.471117,0.469622,286,0.003950,"Low model overfit, but can be economically diluted. Reject: negative cost-adjusted OOS."
11,CORN,equal_family_weight,long_short,True,-0.400244,-183.281621,-534.051632,0.006349,-534.051632,0.470779,290,0.004454,Low model overfit; family choices are researcher-defined. Reject: negative cost-adjusted OOS.
25,SOYABEAN,avg_all_signals,long_short,True,0.382216,165.621529,-342.013053,0.048464,-432.775906,0.491429,258,0.001993,"Low model overfit, but can be economically diluted."
27,SOYABEAN,equal_family_weight,long_short,True,0.233184,105.882051,-351.767290,-0.034708,-488.597670,0.481550,261,0.002182,Low model overfit; family choices are researcher-defined.
40,WHEAT_HRW,avg_all_signals,long_short,True,-0.477294,-207.751409,-359.037180,-0.264086,-723.572172,0.481605,288,0.002488,"Low model overfit, but can be economically diluted. Reject: negative cost-adjusted OOS."
39,WHEAT_HRW,equal_family_weight,long_short,True,-0.302417,-155.124783,-367.588727,-0.286078,-826.259960,0.477093,302,0.002564,Low model overfit; family choices are researcher-defined. Reject: negative cost-adjusted OOS.
55,WHEAT_SRW,avg_all_signals,long_short,True,-0.470679,-216.627499,-419.099848,-0.402158,-733.478916,0.478705,281,0.002534,"Low model overfit, but can be economically diluted. Reject: negative cost-adjusted OOS."
54,WHEAT_SRW,equal_family_weight,long_short,True,-0.358633,-189.586172,-423.458172,-0.411994,-845.450996,0.473684,288,0.002636,Low model overfit; family choices are researcher-defined. Reject: negative cost-adjusted OOS.


## 2. IC Family Selection

Rationale:

If averaging is diluted, test whether predictive families can be selected using train/validation IC.

Important overfit question:

If the validation-selected family performs badly OOS, then IC selection is unstable and should be treated as diagnostic only.

In [7]:
ic_rows = family_results.loc[
    family_results["strategy"].str.contains("ic_family_selected", regex=False),
    [
        "commodity", "strategy", "mode", "cost_adjusted", "oos_sharpe", "oos_pnl", "oos_dd",
        "full_sharpe", "full_dd", "hit_rate", "win_days", "turnover", "overfit_comment",
    ],
].sort_values(["commodity", "oos_sharpe"], ascending=[True, False])

ic_rows

,commodity,strategy,mode,cost_adjusted,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,hit_rate,win_days,turnover,overfit_comment
8,CORN,ic_family_selected_curve,long_short,True,-0.132723,-150.374274,-618.134373,-0.026827,-1578.338865,0.473103,343,0.005800,Moderate/high selection risk; validation IC can be noisy. Reject: negative cost-adjusted OOS.
22,SOYABEAN,ic_family_selected_price,long_short,True,0.721160,667.300247,-684.304850,0.466346,-684.304850,0.501441,348,0.002905,Moderate/high selection risk; validation IC can be noisy.
38,WHEAT_HRW,ic_family_selected_physical_public,long_short,True,-0.240431,-249.674424,-605.622668,0.167225,-1117.274195,0.490806,347,0.004279,Moderate/high selection risk; validation IC can be noisy. Reject: negative cost-adjusted OOS.
52,WHEAT_SRW,ic_family_selected_physical_public,long_short,True,-0.160302,-184.145713,-625.959757,0.403247,-753.866084,0.466763,323,0.004486,Moderate/high selection risk; validation IC can be noisy. Reject: negative cost-adjusted OOS.


## 3. Regime Tests: Best Family vs Average Family

Rationale:

Some signals may work only in specific states. I tested trend/MR regimes in two ways:

1. Select the best family per regime.
2. Average eligible families per regime to reduce selection risk.

The comparison tells me whether regime conditioning helps, and whether family selection adds too much instability.

In [8]:
regime_rows = family_results.loc[
    family_results["strategy"].isin(["regime_best_family_trend_mr", "regime_avg_families_trend_mr"]),
    [
        "commodity", "strategy", "mode", "cost_adjusted", "oos_sharpe", "oos_pnl", "oos_dd",
        "full_sharpe", "full_dd", "hit_rate", "win_days", "turnover", "overfit_comment",
    ],
].sort_values(["commodity", "oos_sharpe"], ascending=[True, False])

regime_rows

,commodity,strategy,mode,cost_adjusted,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,hit_rate,win_days,turnover,overfit_comment
12,CORN,regime_best_family_trend_mr,long_short,True,-0.558358,-227.400880,-460.758024,0.792422,-460.758024,0.462475,228,0.008655,"Moderate selection risk; weights are fixed masks, not optimized. Reject: negative cost-adjusted OOS."
13,CORN,regime_avg_families_trend_mr,long_short,True,-0.705526,-320.335315,-532.754967,0.672769,-532.754967,0.477941,260,0.006356,"Lower than best-family regime selection, but still regime-research risk. Reject: negative cost-adjusted OOS."
21,SOYABEAN,regime_best_family_trend_mr,long_short,True,1.005108,515.192793,-223.005657,0.712843,-456.834512,0.516505,266,0.004299,"Moderate selection risk; weights are fixed masks, not optimized."
23,SOYABEAN,regime_avg_families_trend_mr,long_short,True,0.540224,217.241832,-195.988627,0.278737,-680.558931,0.489170,271,0.003277,"Lower than best-family regime selection, but still regime-research risk."
35,WHEAT_HRW,regime_best_family_trend_mr,long_short,True,0.029644,10.359279,-197.169027,0.210953,-576.778467,0.469582,247,0.005019,"Moderate selection risk; weights are fixed masks, not optimized."
37,WHEAT_HRW,regime_avg_families_trend_mr,long_short,True,-0.142032,-54.906726,-271.863928,0.120905,-462.508561,0.471404,272,0.002916,"Lower than best-family regime selection, but still regime-research risk. Reject: negative cost-adjusted OOS."
49,WHEAT_SRW,regime_avg_families_trend_mr,long_short,True,0.183743,88.211850,-199.602011,0.604021,-395.920862,0.479428,268,0.003202,"Lower than best-family regime selection, but still regime-research risk."
51,WHEAT_SRW,regime_best_family_trend_mr,long_short,True,0.030022,31.946072,-405.187227,0.380509,-680.587284,0.462069,335,0.005069,"Moderate selection risk; weights are fixed masks, not optimized."


## 4. OLS / Kalman / Ridge Benchmark

Rationale:

After testing fixed rules, I tested whether fitted/dynamic coefficients could improve results.

These are benchmarks, not automatically preferred strategies, because they introduce coefficient-estimation risk.

In [9]:
linear_out = run_linear_online_model_experiment(data_dir=DATA_DIR)
linear_results = linear_out["results"].copy()

if COST_ONLY:
    linear_results = linear_results.loc[linear_results["cost_adjusted"].astype(bool)].copy()

linear_summary = linear_results.loc[
    :,
    [
        "commodity", "model", "mode", "cost_adjusted", "test_sharpe", "test_pnl", "test_max_drawdown",
        "full_sharpe", "max_drawdown", "turnover", "avg_gross_exposure",
    ],
].sort_values(["commodity", "test_sharpe"], ascending=[True, False])

linear_summary

KeyError: 'results'

# Part 2: Soybean Research Path

Research logic:

1. Check provided-data soybean signals, especially Cargill crush and Cargill inventory.
2. Test external economic signals: external crush, FX/export, weather HDD/CDD.
3. Diagnose weak periods.
4. Test low-volatility risk controls.

## 5. Soybean Given-Data-Only Tests

Rationale:

The exercise requires using the provided datasets, so I first tested whether provided soybean data alone could work.

Key thing to check:

Does Cargill crush pressure work before adding external signals?

In [10]:
soy_given = run_soybean_given_only_experiment(data_dir=DATA_DIR)
soy_given_results = soy_given["results"].copy()

if COST_ONLY:
    soy_given_results = soy_given_results.loc[soy_given_results["cost_adjusted"].astype(bool)].copy()

soy_given_results[[
    "experiment", "mode", "cost_adjusted", "test_sharpe", "test_pnl", "test_max_drawdown",
    "full_sharpe", "max_drawdown", "turnover",
]].sort_values("test_sharpe", ascending=False)

,experiment,mode,cost_adjusted,test_sharpe,test_pnl,test_max_drawdown,full_sharpe,max_drawdown,turnover
7,given_crush_pressure,long_only,True,1.603698,206.917387,-94.773566,1.862057,-117.766187,0.003755
8,given_trend,long_only,True,1.415391,334.129372,-200.427085,0.739468,-300.869959,0.001426
9,given_conservative_long_blend,long_only,True,1.392714,226.596817,-87.736431,0.950619,-238.302347,0.001658
10,given_price_family,long_only,True,1.383825,436.973678,-217.132103,1.032327,-343.482420,0.001397
11,given_equal_price_physical,long_short,True,0.936097,286.302738,-196.783377,0.398896,-257.449952,0.001488
12,given_physical_family,long_only,True,0.740546,153.365405,-123.622591,0.509009,-143.225412,0.002086
13,given_inventory_pressure,long_only,True,-0.907756,-200.898487,-294.665828,-0.896310,-375.739238,0.002494


## 6. Soybean External Signal Tests

Rationale:

Once Cargill crush worked, I tested economically relevant external families:

- external crush proxy from soybean meal and soybean oil;
- FX/export pressure from USD, BRL, CNY;
- weather HDD/CDD/GDD from crop-belt weather;
- macro/risk proxies.

This is the point where I test whether external data genuinely improves the provided-data baseline.

In [11]:
soy_external = run_soybean_external_signal_experiment(data_dir=DATA_DIR, include_weather=True)
soy_external_results = soy_external["results"].copy()

if COST_ONLY:
    soy_external_results = soy_external_results.loc[soy_external_results["cost_adjusted"].astype(bool)].copy()

soy_external_results[[
    "experiment", "family", "mode", "cost_adjusted", "test_sharpe", "test_pnl", "test_max_drawdown",
    "full_sharpe", "max_drawdown", "turnover",
]].sort_values("test_sharpe", ascending=False).head(15)

,experiment,family,mode,cost_adjusted,test_sharpe,test_pnl,test_max_drawdown,full_sharpe,max_drawdown,turnover
34,given_crush_plus_weather_hdd_cdd_equal_weight,given_crush_weather_equal,long_only,True,2.899920,159.501813,-31.081004,2.162887,-78.107398,0.001866
35,given_physical_external_fundamentals_equal_weight,physical_crush_fx_weather_equal,long_only,True,2.453357,234.780265,-96.223375,1.202564,-144.116601,0.001142
36,regime_candidate_regime_hybrid_vol_trend_weight,observable_regime_shift,long_only,True,2.373358,269.603165,-90.671840,1.563574,-168.867102,0.001009
37,regime_pre2018_selection_regime_hybrid_vol_trend_weight,pre2018_selected_observable_regime_shift,long_only,True,2.373358,269.603165,-90.671840,1.563574,-168.867102,0.001009
38,fund_candidate_fund_blend_65_regime_35_drawdown,fund_usable_regime_drawdown_blend,long_only,True,2.238787,230.063154,-90.378791,1.559172,-135.272524,0.001035
39,fund_pre2018_selection_fund_blend_65_regime_35_drawdown,pre2018_selected_fund_usable_blend,long_only,True,2.238787,230.063154,-90.378791,1.559172,-135.272524,0.001035
40,weight_relaxed_candidate_physical_40_fx_20_extcrush_20_weather_20,predefined_fixed_family_weights,long_only,True,2.204354,223.430010,-55.136056,1.237083,-85.950234,0.001168
41,drawdown_priority_pre2018_selection_physical_40_fx_20_extcrush_20_weather_20,pre2018_drawdown_priority_fixed_family_weights,long_only,True,2.204354,223.430010,-55.136056,1.237083,-85.950234,0.001168
42,fund_candidate_fund_blend_50_regime_50_drawdown,fund_usable_regime_drawdown_blend,long_only,True,2.192012,221.652019,-90.489841,1.527053,-124.879798,0.001046
43,fund_candidate_fund_blend_35_regime_65_drawdown,fund_usable_regime_drawdown_blend,long_only,True,2.052964,202.076759,-91.963167,1.046018,-136.629028,0.001090


## 7. Soybean Low-Volatility Risk-Control Tests

Rationale:

Performance analysis showed soybean weakness in low-volatility / abundant-supply periods. So I tested risk controls rather than adding fitted alpha weights.

Main question:

Is it better to reduce exposure in low volatility instead of replacing the model with a new low-vol alpha?

In [12]:
soy_low_vol = run_soybean_low_vol_switch_experiment(data_dir=DATA_DIR)
soy_low_vol_results = soy_low_vol["results"].copy()

if COST_ONLY:
    soy_low_vol_results = soy_low_vol_results.loc[soy_low_vol_results["cost_adjusted"].astype(bool)].copy()

soy_low_vol_results[[
    "strategy", "cost_adjusted", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "full_dd",
    "low_vol_sharpe", "low_vol_pnl", "low_vol_dd", "low_price_pnl", "low_price_dd", "turnover",
]].sort_values("oos_sharpe", ascending=False)

,strategy,cost_adjusted,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,low_vol_sharpe,low_vol_pnl,low_vol_dd,low_price_pnl,low_price_dd,turnover
13,low_vol_flat_else_base,True,5.056833,103.015642,-29.190664,1.200822,-94.069062,NaN,NaN,NaN,-57.006293,-80.789331,0.000908
14,low_vol_abundant_flat_weak_nonabundant_flat_else_base,True,2.758423,225.818211,-50.456815,1.413593,-92.509563,1.714939,127.177610,-44.328960,-54.854390,-78.637427,0.001474
15,low_vol_abundant_half_weak_nonabundant_flat_else_base,True,2.741884,246.469279,-50.456815,1.339729,-104.046542,1.505129,135.699295,-44.328960,-66.983773,-90.766810,0.001404
16,low_vol_half_base_else_base,True,2.505201,163.222826,-34.702062,1.167465,-85.029854,1.288582,70.357961,-28.200066,-47.967085,-71.750122,0.000795
17,low_vol_weak_trend_flat_else_base,True,2.390410,212.967162,-50.988987,1.150979,-111.779676,1.128318,98.857454,-52.064720,-70.323497,-94.106534,0.001482
18,low_vol_weak_trend_half_else_base,True,2.224077,217.986986,-50.456815,1.166182,-92.053147,1.157654,119.210397,-49.699375,-54.990378,-78.773415,0.001363
19,base_drawdown_priority,True,2.204354,223.430010,-55.136056,1.237083,-85.950234,1.288582,140.715922,-56.400133,-38.927877,-70.740328,0.001168
20,low_vol_weak_physical_flat_else_base,True,2.192134,178.861394,-51.645773,1.109842,-118.265880,0.990636,66.547883,-55.526171,-66.304164,-90.087201,0.001762
21,low_vol_abundant_proxy_half_else_base,True,2.080457,201.899941,-55.136056,1.230017,-80.847887,1.278076,131.130072,-56.400133,-26.983659,-60.835941,0.001288
22,low_vol_abundant_proxy_flat_else_base,True,2.031546,181.209391,-68.009697,1.286889,-88.169771,1.406117,122.467359,-69.273774,-14.955821,-50.847934,0.001372


## 8. Soybean Key-Period Performance

Rationale:

This is the performance-analyst step. It shows whether the improvement is broad or only one headline metric.

In [ ]:
for name, table in soy_low_vol["period_tables"].items():
    print("
" + name)
    display(table)

SyntaxError: EOL while scanning string literal (3956557926.py, line 2)

# Part 3: Corn Research Path

Research logic:

1. Generic provided-data signals were weak for corn.
2. Corn has a specific demand driver: ethanol.
3. Volatility regime worked better than trend/MR regime.
4. Performance analysis showed abundant-supply drawdown.
5. Test an observable abundant-supply guard.

## 9. Corn External / Ethanol Tests

Rationale:

Corn is directly linked to ethanol demand. This tests whether EIA ethanol production/stocks improve the weak generic corn results.

In [14]:
corn_external = run_corn_signal_experiment(data_dir=DATA_DIR, include_weather=True, include_eia=True)
corn_external_results = corn_external["results"].copy()

if COST_ONLY:
    corn_external_results = corn_external_results.loc[corn_external_results["cost_adjusted"].astype(bool)].copy()

corn_external_results[[
    "strategy", "mode", "cost_adjusted", "oos_sharpe", "oos_pnl", "oos_dd",
    "full_sharpe", "full_dd", "turnover",
]].sort_values("oos_sharpe", ascending=False).head(20)

KeyError: "['strategy', 'oos_sharpe', 'oos_pnl', 'oos_dd', 'full_dd'] not in index"

## 10. Corn Abundant-Supply Risk-Control Tests

Rationale:

Corn's best early strategy still had a large low-price abundant-supply drawdown. This tests simple observable guards:

- price below long moving average;
- negative medium-term momentum;
- low/normal volatility;
- curve confirmation.

In [16]:
corn_guard = run_corn_abundant_supply_improvement(data_dir=DATA_DIR)
corn_guard_results = corn_guard["results"].copy()

if COST_ONLY:
    corn_guard_results = corn_guard_results.loc[corn_guard_results["cost_adjusted"].astype(bool)].copy()

corn_guard_results[[
    "strategy", "cost_adjusted", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "full_dd",
    "low_price_pnl", "low_price_dd", "covid_pnl", "covid_dd", "turnover",
]].sort_values("oos_sharpe", ascending=False)

,strategy,cost_adjusted,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,low_price_pnl,low_price_dd,covid_pnl,covid_dd,turnover
11,below_ma_or_negative_mom_flat,True,1.744611,309.892153,-174.158890,0.735936,-297.795118,-197.307916,-218.096951,NaN,NaN,0.008170
12,below_ma_and_negative_mom_flat,True,1.233042,334.579878,-213.721487,0.590131,-424.955692,-202.899577,-362.330920,NaN,NaN,0.006608
13,abundant_low_or_normal_flat,True,1.119316,332.036819,-213.721487,0.451229,-424.955692,-202.899577,-362.330920,NaN,NaN,0.006629
14,abundant_curve_confirmed_flat,True,1.091186,313.784322,-199.492507,0.399924,-412.409411,-204.582277,-372.735477,-38.735521,-64.357175,0.006492
15,below_ma_and_negative_mom_half,True,0.864622,286.961157,-240.700918,0.341551,-473.105042,-154.371036,-367.884757,-19.367761,-32.178588,0.005273
16,below_ma_or_negative_mom_half,True,0.861841,274.381581,-182.857969,0.302826,-383.081176,-149.691748,-296.113683,-19.367761,-32.178588,0.004121
17,abundant_low_or_normal_half,True,0.831980,285.689627,-240.700918,0.292469,-473.105042,-154.371036,-367.884757,-19.367761,-32.178588,0.005458
18,abundant_curve_confirmed_half,True,0.829334,277.132123,-234.873175,0.273552,-468.345441,-155.439177,-377.059926,-38.735521,-64.357175,0.005359
19,base_regime_ic_vol,True,0.676778,241.856651,-268.877118,0.213104,-519.888178,-101.754849,-394.598721,-38.735521,-64.357175,0.005364
20,abundant_low_vol_flat,True,0.652436,222.163332,-284.859453,0.181137,-497.400089,-204.206009,-363.637352,-38.735521,-64.357175,0.006387


## 11. Corn Key-Period Performance

Rationale:

Check whether the abundant-supply guard really improves the weak regime, and whether it creates new damage elsewhere.

In [15]:
for name, table in corn_guard["period_tables"].items():
    print("
" + name)
    display(table)

SyntaxError: EOL while scanning string literal (2931603554.py, line 2)

# Part 4: Wheat SRW / HRW Research Path

Research logic:

1. Standalone SRW and HRW directional models were weak.
2. Family/regime directional tests were still not enough.
3. The economic insight was to trade SRW/HRW relative value instead of outright wheat direction.
4. Pair mean reversion with a small Cargill physical anchor worked better.
5. Trend-aware variants were tested because pair MR underperformed in trend-like periods.

## 12. Wheat Pair Strategy Tests

Rationale:

After rejecting directional wheat, test SRW/HRW relative value.

Signals include:

- pair price mean reversion;
- pair curve;
- pair COT;
- pair public physical;
- pair Cargill physical using both `cgl_inv` and `cgl_crush`;
- trend-aware variants.

In [17]:
wheat_pair = run_wheat_improvement_experiment(data_dir=DATA_DIR)
wheat_results = wheat_pair["results"].copy()

if COST_ONLY:
    wheat_results = wheat_results.loc[wheat_results["cost_adjusted"].astype(bool)].copy()

wheat_results[[
    "strategy", "cost_adjusted", "oos_sharpe", "oos_pnl", "oos_dd", "oos_dd_pct_avg_margin",
    "full_sharpe", "full_dd", "hit_rate", "win_days", "turnover", "formula",
]].sort_values("oos_sharpe", ascending=False).head(20)

,strategy,cost_adjusted,oos_sharpe,oos_pnl,oos_dd,oos_dd_pct_avg_margin,full_sharpe,full_dd,hit_rate,win_days,turnover,formula
20,pair_price_mr_cargill_trend_conflict_flat_cost_control,True,2.145334,26.671254,-16.955673,20.821759,1.461884,-20.245805,0.506667,38,0.004858,"price-MR/Cargill pair signal, but flat when observable SRW/HRW trend strongly conflicts with MR"
21,pair_price_mr_cargill_80_20_pair_trend_cost_control,True,1.290889,36.374086,-29.441327,42.561576,1.098366,-29.441327,0.517647,88,0.003103,"80% price-MR/Cargill pair signal + 20% SRW/HRW price-ratio trend, with fixed cost-control execution"
22,pair_price_mr_cargill_95_05_cost_control,True,1.222445,34.844399,-23.979667,27.742337,1.496643,-23.979667,0.490909,81,0.004954,"95% SRW/HRW price MR + 5% SRW/HRW Cargill physical family, with fixed cost-control execution"
23,pair_price_mr_cargill_90_10_cost_control,True,1.128948,26.689223,-19.909683,22.755886,1.403119,-22.050536,0.478571,67,0.005002,"90% SRW/HRW price MR + 10% SRW/HRW Cargill physical family, with fixed cost-control execution"
24,pair_price_mr_cost_control,True,1.035247,32.965331,-28.323756,32.275173,1.375622,-28.323756,0.472222,85,0.004993,"same signal as pair_price_mr, but halflife 5, threshold 0.12, weekly rebalance, 40 target daily pair vol"
25,pair_price_mr_physical_90_10_cost_control,True,0.896508,21.914390,-20.836313,24.088117,1.236854,-22.247475,0.496552,72,0.004822,"90% SRW/HRW price MR + 10% SRW/HRW total physical family, with fixed cost-control execution"
26,pair_price_mr,True,0.539491,65.746173,-58.180532,75.995588,0.243434,-142.668086,0.500911,275,0.008666,signal = SRW.rev_5 - HRW.rev_5
27,pair_cot,True,0.100271,8.756899,-46.607109,73.880246,-0.125224,-103.957149,0.481188,243,0.004461,signal = SRW COT family - HRW COT family
28,pair_mr_curve_physical_cost_control,True,0.038225,1.263508,-36.644226,49.617876,0.453674,-68.411073,0.479070,103,0.002024,"same signal as pair_mr_curve_physical_equal, but halflife 5, threshold 0.12, weekly rebalance, 40 target daily pair vol"
29,pair_fixed_trend_mr_regime,True,-0.056777,-5.655212,-64.342796,83.863870,0.068593,-191.872737,0.479245,254,0.004091,if observable trend regime: trend/curve/COT pair; otherwise MR/curve/physical pair


## 13. Wheat Key-Period Performance

Rationale:

Pair MR worked overall, but performance analysis showed weakness in trend-like periods. This cell compares the base pair, the trend-aware blend, and the trend-conflict flat filter.

In [18]:
for key in [
    "pair_price_mr_cargill_90_10_cost_control_cost",
    "pair_price_mr_cargill_80_20_pair_trend_cost_control_cost",
    "pair_price_mr_cargill_trend_conflict_flat_cost_control_cost",
]:
    if key in wheat_pair["backtests"]:
        print("
" + key)
        display(period_performance(wheat_pair["backtests"][key])[["period", "total_pnl", "sharpe", "max_drawdown", "hit_rate", "days"]])

SyntaxError: EOL while scanning string literal (3985909925.py, line 7)

# Part 5: Research Decision Summary

This final cell does not select the highest number mechanically. It summarizes what each stage taught.

In [19]:
research_decisions = pd.DataFrame([
    {
        "Product": "Soybeans",
        "What generic tests showed": "Provided Cargill crush worked; broad/fitted models were not better.",
        "Weakness found by performance analyst": "Low-vol / abundant-supply periods.",
        "Product-specific test justified": "External crush + FX/export + weather, then low-vol half exposure.",
        "Overfit read": "Lower-overfit because final improvement is fixed economic weighting plus observable risk control."
    },
    {
        "Product": "Corn",
        "What generic tests showed": "Average/equal/IC/fitted models were weak; trend/MR was not the right split.",
        "Weakness found by performance analyst": "Large low-price abundant-supply drawdown.",
        "Product-specific test justified": "EIA ethanol and abundant-supply flat guard.",
        "Overfit read": "Improved but still satellite-sized; risk guard is observable, not date-fitted."
    },
    {
        "Product": "Wheat SRW/HRW",
        "What generic tests showed": "Standalone directional SRW and HRW were weak.",
        "Weakness found by performance analyst": "Outright wheat was poor; pair MR weak in trend-like periods.",
        "Product-specific test justified": "SRW/HRW pair MR + Cargill physical; trend-aware variant.",
        "Overfit read": "Cleaner than directional wheat because it uses relative-value economics and no fitted coefficients."
    },
])
research_decisions

,Product,What generic tests showed,Weakness found by performance analyst,Product-specific test justified,Overfit read
0,Soybeans,Provided Cargill crush worked; broad/fitted models were not better.,Low-vol / abundant-supply periods.,"External crush + FX/export + weather, then low-vol half exposure.",Lower-overfit because final improvement is fixed economic weighting plus observable risk control.
1,Corn,Average/equal/IC/fitted models were weak; trend/MR was not the right split.,Large low-price abundant-supply drawdown.,EIA ethanol and abundant-supply flat guard.,"Improved but still satellite-sized; risk guard is observable, not date-fitted."
2,Wheat SRW/HRW,Standalone directional SRW and HRW were weak.,Outright wheat was poor; pair MR weak in trend-like periods.,SRW/HRW pair MR + Cargill physical; trend-aware variant.,Cleaner than directional wheat because it uses relative-value economics and no fitted coefficients.


# Files Created / Updated By These Experiments

Main logs:

- `notes/family_regime_model_comparison.txt`
- `notes/linear_online_models_corn_soybean.txt`
- `notes/soybeans.txt`
- `notes/soybean_low_vol_switch.txt`
- `notes/corn.txt`
- `notes/corn_abundant_supply_improvement.txt`
- `notes/wheat_improvement.txt`
- `notes/strategy_experiment_log.txt`